# MUP by-Provider — Delaware peek

Scope: per-NPI Medicare totals (volume + spend). Primary source for Q2/Q3/Q5.
Source: `data/sources/utilization_and_payment/mup/Medicare_Physician_Other_Practitioners_by_Provider_2024.zip` (~568 MB).
Reading: chunked pandas, filtered to `Rndrng_Prvdr_State_Abrvtn` = DE.

In [1]:
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ZIP = ROOT / "data/sources/utilization_and_payment/mup/Medicare_Physician_Other_Practitioners_by_Provider_2024.zip"
MEMBER = "Medicare_Physician_Other_Practitioners_by_Provider_2024.csv"
STATE_COL, STATE, ENC = "Rndrng_Prvdr_State_Abrvtn", "DE", "latin-1"
KEEP = [
    "Rndrng_NPI", "Rndrng_Prvdr_State_Abrvtn", "Rndrng_Prvdr_Type",
    "Rndrng_Prvdr_Crdntls", "Rndrng_Prvdr_Ent_Cd",
    "Tot_Benes", "Tot_Srvcs", "Tot_Mdcr_Alowd_Amt", "Tot_Mdcr_Pymt_Amt",
    "Drug_Sprsn_Ind", "Med_Sprsn_Ind",
]
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    sample = pd.read_csv(f, nrows=5, dtype=str, encoding=ENC)
print(sample.shape[1], "columns")
sample[KEEP].head()

81 columns


,Rndrng_NPI,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Tot_Benes,Tot_Srvcs,Tot_Mdcr_Alowd_Amt,Tot_Mdcr_Pymt_Amt,Drug_Sprsn_Ind,Med_Sprsn_Ind
0,1003000126,MD,Internal Medicine,M.D.,I,328,399,46472.65,35325.46,NaN,NaN
1,1003000134,IL,Pathology,M.D.,I,912,2023,69422.59,50178.11,NaN,NaN
2,1003000142,OH,Anesthesiology,M.D.,I,316,1369,133096.84,101577.94,NaN,NaN
3,1003000423,OH,Obstetrics & Gynecology,M.D.,I,75,140,8708.74,7134.32,NaN,NaN
4,1003000480,CO,General Surgery,MD,I,101,136,24416.29,18766.8,NaN,NaN


In [2]:
# Scope to Delaware. Chunked scan (~568 MB), key columns only.
parts = []
with zipfile.ZipFile(ZIP) as z, z.open(MEMBER) as f:
    for chunk in pd.read_csv(f, usecols=KEEP, dtype=str, chunksize=200_000, encoding=ENC):
        parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(4931, 11)

In [3]:
# Volume, spend, suppression, provider-type mix.
pay = pd.to_numeric(de["Tot_Mdcr_Pymt_Amt"], errors="coerce")
print("rows:", len(de), "| unique NPIs:", de["Rndrng_NPI"].nunique())
print("total Medicare payment $:", f"{pay.sum():,.0f}")
print("suppressed rows (Med_Sprsn_Ind == '*'):", (de["Med_Sprsn_Ind"] == "*").sum())
de["Rndrng_Prvdr_Type"].value_counts().head(20)

rows: 4931 | unique NPIs: 4931
total Medicare payment $: 555,158,416
suppressed rows (Med_Sprsn_Ind == '*'): 0


Rndrng_Prvdr_Type
Nurse Practitioner                               780
Physician Assistant                              491
Physical Therapist in Private Practice           472
Internal Medicine                                390
Family Practice                                  294
Certified Registered Nurse Anesthetist (CRNA)    219
Emergency Medicine                               166
Chiropractic                                     154
Optometry                                        113
Licensed Clinical Social Worker                  111
Diagnostic Radiology                             102
Anesthesiology                                    83
Obstetrics & Gynecology                           82
General Surgery                                   77
Cardiology                                        75
Mass Immunizer Roster Biller                      71
Psychiatry                                        70
Orthopedic Surgery                                65
Occupational Therapist in Pr

## Notes
- Grain: one row per NPI per year, all services aggregated. `Rndrng_Prvdr_Type` drives the physician-vs-NP/PA split (Q3).
- Spend = `Tot_Mdcr_Pymt_Amt`; separate drug vs medical via `Drug_*`/`Med_*` before ranking (drug skew).
- Suppression: rows with < 11 beneficiaries carry `*` in `Drug_Sprsn_Ind`/`Med_Sprsn_Ind`. Totals will not fully reconcile.